In [5]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx

# 设置H2分子的几何构型
bond_length = 1.4  # H2平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')

# 进行Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

# 进行FCI计算作为参考
cisolver = fci.FCI(mf)
E_fci, fcivec = cisolver.kernel()
print(f"FCI能量: {E_fci:.8f} Ha")

# 使用NetKet创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

Hartree-Fock能量: -0.94148065 Ha
FCI能量: -1.01546825 Ha


In [8]:
import openfermion as of
import openfermionpyscf as ofpyscf
import numpy as np
from pyscf import gto, scf

# ===================== 1. 定义H2分子（与之前一致） =====================
# H2分子几何构型（平衡键长1.4埃）
bond_length = 1.4  # 单位：埃（Å）
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]

# 创建PySCF分子对象（STO-3G基组）
mol = gto.M(
    atom=geometry,
    basis='STO-3G',
    charge=0,
    spin=0,  # H2自旋单重态
    verbose=0
)

# 基础HF计算（为后续哈密顿量构建提供参考）
mf = scf.RHF(mol).run(verbose=0)

# ===================== 2. 构建H2的一次量子化哈密顿量 =====================
# 方法1：从PySCF分子直接构建一次量子化哈密顿量（推荐）
# 输出：MolecularFirstQuantizedHamiltonian对象（包含动能、核吸引、电子排斥项）
hamiltonian_first_q = ofpyscf.generate_molecular_hamiltonian(
    geometry=geometry,
    basis='STO-3G',
    charge=0,
    spin=0,
    n_active_electrons=2,  # H2有2个电子
    n_active_orbitals=2,   # STO-3G基组下H2有2个空间轨道
    format='first_quantized'  # 指定输出一次量子化形式
)

# 方法2：手动拆解一次量子化哈密顿量的各项（直观理解）
# 2.1 动能项（Kinetic Energy）：-1/2 ∑∇ᵢ²
kinetic = of.KineticEnergy()
kinetic_coeff = -0.5  # 原子单位下的动能系数

# 2.2 核吸引项（Nuclear Attraction）：-∑ Zₐ / |rᵢ - Rₐ|
nuclear_attraction = of.NuclearAttraction(
    nuclear_charges=[1.0, 1.0],  # 两个H原子，核电荷数均为1
    nuclear_positions=np.array([[0.,0.,0.], [bond_length,0.,0.]])  # 核位置
)

# 2.3 电子-电子排斥项（Electron-Electron Repulsion）：∑ 1/|rᵢ - rⱼ| (i<j)
electron_repulsion = of.ElectronElectronInteraction()

# 2.4 组合完整一次量子化哈密顿量
hamiltonian_manual = kinetic_coeff * kinetic + nuclear_attraction + electron_repulsion

# ===================== 3. 输出一次量子化哈密顿量的核心信息 =====================
print("=== H2 一次量子化哈密顿量核心信息 ===")
print(f"\n1. 哈密顿量类型: {type(hamiltonian_first_q)}")
print(f"2. 电子数: {hamiltonian_first_q.n_electrons}")
print(f"3. 空间轨道数: {hamiltonian_first_q.n_orbitals}")

# 输出哈密顿量的数学表达式（原子单位）
print("\n4. 一次量子化哈密顿量数学形式（原子单位）:")
print("Ĥ = 动能项 + 核吸引项 + 电子排斥项")
print("   = -1/2 ∑(i=1到2) ∇ᵢ²  - ∑(i=1到2) [1/|rᵢ - R₁| + 1/|rᵢ - R₂|]  + 1/|r₁ - r₂|")

# 输出各项的矩阵元（小体系可直接计算）
print("\n5. 动能项矩阵元（示例）:")
kinetic_matrix = kinetic.to_spatial_basis(hamiltonian_first_q.basis)
print(f"   矩阵形状: {kinetic_matrix.shape}")
print(f"   矩阵元:\n {kinetic_matrix[:2, :2]}")

print("\n6. 核吸引项矩阵元（示例）:")
nuclear_matrix = nuclear_attraction.to_spatial_basis(hamiltonian_first_q.basis)
print(f"   矩阵形状: {nuclear_matrix.shape}")
print(f"   矩阵元:\n {nuclear_matrix[:2, :2]}")

# ===================== 4. 一次量子化 → 二次量子化转换（可选，对比验证） =====================
# 验证：将一次量子化哈密顿量转为二次量子化形式（与之前NetKet结果对齐）
# hamiltonian_second_q = of.get_second_q_ops(hamiltonian_first_q)
# print("\n7. 一次量子化转二次量子化后的形式（前5项）:")
# for term, coeff in list(hamiltonian_second_q.terms.items())[:5]:
#     print(f"   项: {term}, 系数: {coeff.real:.6f}")

TypeError: generate_molecular_hamiltonian() got an unexpected keyword argument 'spin'

In [3]:
ha

ParticleNumberAndSpinConservingFermioperator2nd(_hilbert=SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1)), _operator_data={'diag': {(0, ()): (Array(0, dtype=int32), Array([], shape=(1, 1, 0), dtype=int32), Array([0.37798372], dtype=float64)), (2, (0, 1)): (None, None, COOArray(_coords=Array([[0],
       [1]], dtype=int64), data=Array([-0.94214155, -0.6584201 ], dtype=float64), shape=(2,), fill_value=0)), (4, (0, 1)): (None, None, COOArray(_coords=Array([[1, 0]], dtype=int64), data=Array([-0.34715], dtype=float64), shape=(2, 2), fill_value=0))}, 'offdiag': {}, 'mixed_diag': {(4, ((1, 0),)): (None, None, Array([[0.56481873, 0.57017208],
       [0.57017208, 0.59564759]], dtype=float64))}, 'mixed_offdiag': {(4, ((1, 0),)): (Array([[1, 2],
       [3, 4]], dtype=int64), Array([[[0, 0]],

       [[1, 1]],

       [[1, 0]],

       [[0, 1]],

       [[0, 0]]], dtype=int64), Array([[0.        ],
       [0.22302209],
       [0.22302209],
       [0.22302209],
   

In [4]:
ha

ParticleNumberAndSpinConservingFermioperator2nd(_hilbert=SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions=2, n_fermions_per_spin=(1, 1)), _operator_data={'diag': {(0, ()): (Array(0, dtype=int32), Array([], shape=(1, 1, 0), dtype=int32), Array([0.37798372], dtype=float64)), (2, (0, 1)): (None, None, COOArray(_coords=Array([[0],
       [1]], dtype=int64), data=Array([-0.94214155, -0.6584201 ], dtype=float64), shape=(2,), fill_value=0)), (4, (0, 1)): (None, None, COOArray(_coords=Array([[1, 0]], dtype=int64), data=Array([-0.34715], dtype=float64), shape=(2, 2), fill_value=0))}, 'offdiag': {}, 'mixed_diag': {(4, ((1, 0),)): (None, None, Array([[0.56481873, 0.57017208],
       [0.57017208, 0.59564759]], dtype=float64))}, 'mixed_offdiag': {(4, ((1, 0),)): (Array([[1, 2],
       [3, 4]], dtype=int64), Array([[[0, 0]],

       [[1, 1]],

       [[1, 0]],

       [[0, 1]],

       [[0, 0]]], dtype=int64), Array([[0.        ],
       [0.22302209],
       [0.22302209],
       [0.22302209],
   

SpinOrbitalFermions 是 NetKet（特别是 netket ≥ v3.10 之后，或通过 netket.experimental）中用于描述具有自旋的费米子系统的希尔伯特空间（Hilbert space）类。它专为处理电子结构问题（如分子、固体中的多电子系统）而设计，天然支持泡利不相容原理和固定粒子数约束。

SpinOrbitalFermions 定义了一个由自旋轨道占据数构成的离散希尔伯特空间，每个自旋轨道只能被占据（1）或空（0），且满足：

费米子统计（自动通过占据数表示处理）
可选：固定总电子数、固定自旋向上/向下电子数


In [2]:
hilbert = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,  # 总空间轨道数
    s = 1/2,
    n_fermions_per_spin=(1, 1)  # 每种自旋的电子数
)

# 创建采样器 - 使用费米子跳跃采样器
# 对于分子系统，我们使用完整的轨道图（完全连接）
# cluster = [(0,1),(2,3)]
#
# g = nk.graph.Graph(edges=[(0,2),(1,3),(2,0),(3,1)])
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sa = nk.sampler.MetropolisFermionHop(
    hilbert, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

In [5]:
# 2.1 自回归神经网络模型（适配4个组态的小体系）
ar_model = nk.models.ARNNSequential(
    hilbert=hilbert,
    #param_dtype=complex,  # 复参数适配VQMC的holomorphic优化
)

# 设置machine_pow属性（用于VQMC的|ψ|²采样）
ar_model.machine_pow = 2

# 2.2 自回归直接采样器（无拒绝、无burn-in，完美适配4个组态）
# 注意：ARDirectSampler只能用于无约束的希尔伯特空间
# 由于我们使用了 n_fermions_per_spin 约束，需要创建无约束空间
hilbert_unconstrained = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    # 不设置 n_fermions_per_spin，创建无约束空间
)

ar_sampler = nk.sampler.ARDirectSampler(
    hilbert_unconstrained,
    dtype=complex
)

In [8]:
hilbert.all_states()

Array([[0, 1, 0, 1],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [1, 0, 1, 0]], dtype=int8)

In [ ]:
# 创建变分量子态 - 使用Slater行列式作为初始波函数
ma = nk.models.RBM(alpha=1, param_dtype=complex, use_visible_bias=False)
vs = nk.vqs.MCState(sa, ma, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01, holomorphic=True)

# 创建VMC驱动器
gs = nk.driver.VMC(ha, opt, variational_state=vs, preconditioner=sr)

# 运行优化
exp_name = "h2_molecule"

In [ ]:
gs.run(300, out=exp_name)

In [ ]:
############## 绘图 #################

# 获取精确对角化能量（FCI能量）
ed_energies = np.array([E_fci])  # H2只有一个基态能量

# 读取日志数据
with open(f"{exp_name}.log") as f:
    data = json.load(f)

x = data["Energy"]["iters"]
y = data["Energy"]["Mean"]['real']

# 绘制能量收敛曲线
plt.figure(figsize=(10, 6))
plt.axhline(ed_energies[0], color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.plot(x, y, 'b-', label="VMC energy")
plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence")
plt.legend()
plt.grid(True)
plt.show()

# 打印最终结果
print(f"最终VMC能量: {y[-1]:.8f} Ha")
print(f"最终FCI能量: {E_fci:.8f} Ha")
print(f"最终Hartree-Fock能量: {E_hf:.8f} Ha")
print(f"与FCI能量误差: {abs(y[-1] - E_fci):.8f} Ha")
